In [1]:
# pipeline base params
table_name = "payroll_data_silver"

StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re

StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 4, Finished, Available, Finished, False)

In [3]:
payroll_2020 = spark.read.format("csv").option("header","true").load("abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/nycpayroll_2020.csv")
# df now is a Spark DataFrame containing CSV data from "abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/nycpayroll_2020.csv".
display(payroll_2020)

StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b9c79bf-5f3c-4d84-b53b-373775645736)

In [4]:
payroll_2021 = spark.read.format("csv").option("header","true").load("abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/nycpayroll_2021.csv")
# df now is a Spark DataFrame containing CSV data from "abfss://dev_Bronze@onelake.dfs.fabric.microsoft.com/lh_nyc_payroll_bronze.Lakehouse/Files/raw/nycpayroll_2021.csv".
display(payroll_2021)

StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ea2aa90e-c2a0-41c6-8f89-7b1b6dfcb036)

In [ ]:
def standardize_columns(df):
    for c in df.columns:
        new_c = c.lower().strip().replace(" ", "_")
        new_c = re.sub(r'[^a-z0-9_]', '', new_c)
        df = df.withColumnRenamed(c, new_c)
    return df

# Standardize
payroll_2020 = standardize_columns(payroll_2020)
payroll_2021 = standardize_columns(payroll_2021)

# Align agency column name
if "agencycode" in payroll_2021.columns:
    payroll_2021 = payroll_2021.withColumnRenamed("agencycode", "agencyid")

# Remove duplicates
payroll_2020 = payroll_2020.dropDuplicates()
payroll_2021 = payroll_2021.dropDuplicates()

# Handle nulls
payroll_2020 = payroll_2020.filter(F.col("employeeid").isNotNull())
payroll_2021 = payroll_2021.filter(F.col("employeeid").isNotNull())

# Numeric columns to clean
numeric_cols = [
    "basesalary",
    "regularhours",
    "regulargrosspaid",
    "othours",
    "totalotpaid",g
    "totalotherpay"
]

# Clean numeric columns properly
for c in numeric_cols:
    if c in payroll_2020.columns:
        payroll_2020 = payroll_2020.fillna({c: 0}) \
                                   .withColumn(c, F.col(c).cast(DoubleType()))

    if c in payroll_2021.columns:
        payroll_2021 = payroll_2021.fillna({c: 0}) \
                                   .withColumn(c, F.col(c).cast(DoubleType()))

# Union payroll
payroll_df = payroll_2020.unionByName(payroll_2021, allowMissingColumns=True)

#Drop duplicate columns from payroll
payroll_df = payroll_df.drop("firstname", "lastname")

# Flag outlier
payroll_df = payroll_df.withColumn(
    "fiscalyear_outlier_flag",
    F.when(
        (F.col("fiscalyear") < 2020) | (F.col("fiscalyear") > 2021),
        True
    ).otherwise(False)
)


display(payroll_df)

StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 177b894f-fb96-401d-b55a-8efdf945ab34)

In [6]:
# Write Silver
payroll_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("Silver Payroll completed")


StatementMeta(, bb7368c0-a213-442f-ac42-15d2940db97e, 8, Finished, Available, Finished, False)

Silver Payroll completed
